#### 1. Local environment test

In [3]:
#!/usr/bin/env python3
"""
compare_predictions.py

Compare ground truth and WaveStitch+ predictions over all target features.
Includes:
1. Individual feature plots
2. Grid plot
3. Grouped plots
4. Interactive zoom helper
"""

from __future__ import annotations

import json
from pathlib import Path
from typing import Optional

import matplotlib
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ─────────────────────────────────────────────
# Global academic-style plotting configuration
# ─────────────────────────────────────────────
matplotlib.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 10,
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 8,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#333333",
    "axes.linewidth": 0.8,
    "axes.grid": True,
    "grid.color": "#D9D9D9",
    "grid.linestyle": "--",
    "grid.linewidth": 0.5,
    "grid.alpha": 0.6,
    "savefig.facecolor": "white",
    "savefig.bbox": "tight",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.unicode_minus": False,
})

COLORS = {
    "gt": "#1f77b4",              # muted blue
    "wavestitchplus": "#ff7f0e",            # orange
}


SHOW_PLOTS = False
SAVE_PNG = True
SAVE_PDF = True
PNG_DPI = 300

# ─────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────
# DATASET = "python"
# DATASET = "golang"
# DATASET = "amf"
DATASET = "rabbitmq"

BASE_DIR = Path("./work/EUR")
PREPARED_DIR = BASE_DIR / f"prepared_{DATASET}"
GENERATED_DIR = BASE_DIR / f"generated_{DATASET}"
OUTPUT_DIR = GENERATED_DIR / "plots"

PRED_FILE = "wavestitchPlus_full_imputed.csv"


# ─────────────────────────────────────────────
# Data loading
# ─────────────────────────────────────────────

def load_meta(prepared_dir: Path) -> dict:
    meta_path = prepared_dir / "meta.json"
    if not meta_path.exists():
        raise FileNotFoundError(f"Missing file: {meta_path}")
    with meta_path.open("r", encoding="utf-8") as f:
        return json.load(f)


def load_tables(prepared_dir: Path, generated_dir: Path, pred_file: str):
    gt_path = prepared_dir / "test_gt.csv"
    pred_path = generated_dir / pred_file

    if not gt_path.exists():
        raise FileNotFoundError(f"Missing file: {gt_path}")
    if not pred_path.exists():
        raise FileNotFoundError(f"Missing file: {pred_path}")

    gt = pd.read_csv(gt_path)
    pred = pd.read_csv(pred_path)
    return gt, pred


# ─────────────────────────────────────────────
# Time parsing
# ─────────────────────────────────────────────

def parse_time_column(df: pd.DataFrame, time_col: str) -> pd.DatetimeIndex:
    """
    Parse time column robustly:
    - Unix timestamp (seconds / milliseconds)
    - datetime strings
    - fallback to synthetic index time
    """
    if time_col not in df.columns:
        print(f"[WARNING] Time column '{time_col}' not found, using synthetic index")
        return pd.to_datetime(range(len(df)), unit="s")

    time_data = df[time_col]
    valid = time_data.dropna()
    sample = valid.iloc[0] if len(valid) > 0 else None
    print(f"[INFO] Time sample: {sample} (type: {type(sample).__name__})")

    try:
        if pd.api.types.is_numeric_dtype(time_data):
            max_val = valid.max() if len(valid) > 0 else 0
            if max_val > 1e12:
                print("[INFO] Detected Unix timestamp (milliseconds)")
                parsed = pd.to_datetime(time_data, unit="ms", errors="coerce")
            else:
                print("[INFO] Detected Unix timestamp (seconds)")
                parsed = pd.to_datetime(time_data, unit="s", errors="coerce")
        else:
            print("[INFO] Trying to parse as datetime string")
            parsed = pd.to_datetime(time_data, errors="coerce")

        if pd.Series(parsed).isna().all():
            print("[WARNING] All parsed times are NaT, using synthetic index")
            return pd.to_datetime(range(len(df)), unit="s")

        return parsed

    except Exception as e:
        print(f"[WARNING] Failed to parse time: {e}, using synthetic index")
        return pd.to_datetime(range(len(df)), unit="s")


def setup_time_axis(ax: plt.Axes, time_index: pd.DatetimeIndex) -> None:
    ts = pd.Series(time_index).dropna()
    if len(ts) == 0:
        return

    duration = ts.max() - ts.min()

    if duration.days > 30:
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
        ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
    elif duration.days > 1:
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
        ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
    elif duration.total_seconds() > 3600:
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
        ax.xaxis.set_major_locator(mdates.HourLocator(interval=1))
    else:
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
        ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=5))

    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")


# ─────────────────────────────────────────────
# Plot saving helper
# ─────────────────────────────────────────────

def save_figure(fig: plt.Figure, path_base: Path) -> None:
    if SAVE_PNG:
        png_path = path_base.with_suffix(".png")
        fig.savefig(png_path, dpi=PNG_DPI)
        print(f"[SAVED] {png_path}")
    if SAVE_PDF:
        pdf_path = path_base.with_suffix(".pdf")
        fig.savefig(pdf_path)
        print(f"[SAVED] {pdf_path}")


# ─────────────────────────────────────────────
# Plot 1: Individual feature plots
# ─────────────────────────────────────────────

def plot_individual_features(
    gt: pd.DataFrame,
    pred_ws_plus: pd.DataFrame,
    target_cols: list[str],
    time_index: pd.DatetimeIndex,
    output_dir: Path,
    dataset_name: str,
) -> None:
    print("\n[PLOTTING] Individual feature plots...")

    for feature_name in target_cols:
        if feature_name not in gt.columns:
            print(f"[SKIP] {feature_name} not in ground truth")
            continue

        fig, ax = plt.subplots(figsize=(15, 4.2))

        ax.plot(
            time_index,
            gt[feature_name].to_numpy(),
            label="Ground truth",
            alpha=0.9,
            color=COLORS["gt"],
            linewidth=1.3,
        )

        if feature_name in pred_ws_plus.columns:
            ax.plot(
                time_index,
                pred_ws_plus[feature_name].to_numpy(),
                label="WaveStitch+",
                alpha=0.9,
                color=COLORS["wavestitchplus"],
                linewidth=1.3,
            )

        ax.set_ylabel(feature_name)
        ax.set_xlabel("Time")
        ax.legend(frameon=True, loc="upper right")
        setup_time_axis(ax, time_index)

        plt.tight_layout()
        save_figure(fig, output_dir / feature_name)

        if SHOW_PLOTS:
            plt.show()
        plt.close(fig)


# ─────────────────────────────────────────────
# Plot 2: Grid plot
# ─────────────────────────────────────────────

def plot_grid(
    gt: pd.DataFrame,
    pred_ws_plus: pd.DataFrame,
    target_cols: list[str],
    time_index: pd.DatetimeIndex,
    output_dir: Path,
    dataset_name: str,
    n_cols: int = 3,
) -> None:
    print("\n[PLOTTING] Grid plot...")

    valid_features = [f for f in target_cols if f in gt.columns]
    if not valid_features:
        print("[WARNING] No valid target columns for grid plot")
        return

    n_features = len(valid_features)
    n_rows = (n_features + n_cols - 1) // n_cols

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(18, 3.1 * n_rows),
        squeeze=False,
    )
    axes = axes.flatten()

    for idx, feature_name in enumerate(valid_features):
        ax = axes[idx]

        ax.plot(
            time_index,
            gt[feature_name].to_numpy(),
            label="GT",
            alpha=0.9,
            color=COLORS["gt"],
            linewidth=1.0,
        )

        if feature_name in pred_ws_plus.columns:
            ax.plot(
                time_index,
                pred_ws_plus[feature_name].to_numpy(),
                label="WS+",
                alpha=0.9,
                color=COLORS["wavestitchplus"],
                linewidth=1.0,
            )

        ax.set_title(feature_name, pad=4)
        ax.legend(frameon=True, loc="upper right", fontsize=7)

        # Simpler axis formatting for compact grid
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right", fontsize=7)

    for idx in range(len(valid_features), len(axes)):
        axes[idx].set_visible(False)

    fig.suptitle(f"{dataset_name} — All Target Features", y=1.01, fontsize=12)
    plt.tight_layout()
    save_figure(fig, output_dir / "all_features_grid")

    if SHOW_PLOTS:
        plt.show()
    plt.close(fig)


# ─────────────────────────────────────────────
# Plot 3: Grouped plots
# ─────────────────────────────────────────────

def auto_group_features(feature_list: list[str]) -> dict[str, list[str]]:
    """Automatically group features by semantic keywords."""
    groups = {
        "CPU": [],
        "Memory": [],
        "Latency": [],
        "Load": [],
        "Other": [],
    }

    for f in feature_list:
        f_lower = f.lower()
        if "cpu" in f_lower:
            groups["CPU"].append(f)
        elif "ram" in f_lower or "mem" in f_lower:
            groups["Memory"].append(f)
        elif "lat" in f_lower:
            groups["Latency"].append(f)
        elif f_lower in ["n", "c", "qps", "rps", "requests"]:
            groups["Load"].append(f)
        else:
            groups["Other"].append(f)

    return {k: v for k, v in groups.items() if v}


def plot_grouped(
    gt: pd.DataFrame,
    pred_ws_plus: pd.DataFrame,
    target_cols: list[str],
    time_index: pd.DatetimeIndex,
    output_dir: Path,
    dataset_name: str,
) -> None:
    feature_groups = auto_group_features(target_cols)

    print("\n[PLOTTING] Grouped plots...")
    for group_name, features in feature_groups.items():
        features = [f for f in features if f in gt.columns]
        if not features:
            continue

        n_features = len(features)
        fig, axes = plt.subplots(
            n_features, 1,
            figsize=(15, 2.6 * n_features),
            sharex=True,
            squeeze=False,
        )
        axes = axes.flatten()

        for ax, feature_name in zip(axes, features):
            ax.plot(
                time_index,
                gt[feature_name].to_numpy(),
                label="Ground truth",
                alpha=0.9,
                color=COLORS["gt"],
                linewidth=1.2,
            )

            if feature_name in pred_ws_plus.columns:
                ax.plot(
                    time_index,
                    pred_ws_plus[feature_name].to_numpy(),
                    label="WaveStitch+",
                    alpha=0.9,
                    color=COLORS["wavestitchplus"],
                    linewidth=1.2,
                )

            ax.set_ylabel(feature_name)
            ax.legend(loc="upper right", frameon=True)

        setup_time_axis(axes[-1], time_index)
        axes[-1].set_xlabel("Time")

        plt.tight_layout()
        save_figure(fig, output_dir / f"group_{group_name.lower()}")

        if SHOW_PLOTS:
            plt.show()
        plt.close(fig)


# ─────────────────────────────────────────────
# Plot 4: Interactive helper
# ─────────────────────────────────────────────

def plot_interactive(
    feature_name: str,
    gt: pd.DataFrame,
    pred_ws_plus: pd.DataFrame,
    time_index: pd.DatetimeIndex,
    dataset_name: str,
    start_time: Optional[str] = None,
    end_time: Optional[str] = None,
) -> None:
    """
    Plot a selected feature in a chosen time range.

    Example:
        plot_interactive("cpu_usage", gt, pred_ws_plus, time_index, DATASET)
        plot_interactive("cpu_usage", gt, pred_ws_plus, time_index, DATASET,
                         "2022-03-16 10:00", "2022-03-16 12:00")
    """
    if feature_name not in gt.columns:
        print(f"[ERROR] {feature_name} not found in ground truth")
        return

    df_plot = pd.DataFrame({
        "time": time_index,
        "Ground truth": gt[feature_name].to_numpy(),
    })

    if feature_name in pred_ws_plus.columns:
        df_plot["WaveStitch+"] = pred_ws_plus[feature_name].to_numpy()

    df_plot = df_plot.set_index("time")

    if start_time:
        df_plot = df_plot[df_plot.index >= pd.to_datetime(start_time)]
    if end_time:
        df_plot = df_plot[df_plot.index <= pd.to_datetime(end_time)]

    fig, ax = plt.subplots(figsize=(15, 5))
    df_plot.plot(
        ax=ax,
        linewidth=1.3,
        color=[COLORS["gt"], COLORS["wavestitchplus"][:]] if "WaveStitch+" in df_plot.columns else [COLORS["gt"]],
    )

    ax.set_title(feature_name, pad=6)
    ax.set_xlabel("Time")
    ax.set_ylabel(feature_name)
    ax.legend(frameon=True)
    setup_time_axis(ax, df_plot.index)

    plt.tight_layout()
    if SHOW_PLOTS:
        plt.show()
    plt.close(fig)


# ─────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────

def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    meta = load_meta(PREPARED_DIR)
    time_col = meta.get("time_col", "time")
    target_cols = meta.get("target_cols", [])
    cond_cols = meta.get("cond_cols", [])

    print(f"{'='*60}")
    print(f"Loaded meta from: {PREPARED_DIR / 'meta.json'}")
    print(f"{'='*60}")
    print(f"Time column: {time_col}")
    print(f"Target columns ({len(target_cols)}): {target_cols}")
    print(f"Conditioning columns ({len(cond_cols)}): {cond_cols}")
    print(f"{'='*60}\n")

    gt, pred_ws_plus = load_tables(PREPARED_DIR, GENERATED_DIR, PRED_FILE)
    print(f"Data shapes: GT={gt.shape}, WaveStitch+={pred_ws_plus.shape}")

    time_index = parse_time_column(gt, time_col)
    ts = pd.Series(time_index).dropna()
    if len(ts) > 0:
        print(f"[INFO] Time range: {ts.min()} to {ts.max()}")
        print(f"[INFO] Duration: {ts.max() - ts.min()}")
    else:
        print("[WARNING] Invalid time range")

    plot_individual_features(
        gt=gt,
        pred_ws_plus=pred_ws_plus,
        target_cols=target_cols,
        time_index=time_index,
        output_dir=OUTPUT_DIR,
        dataset_name=DATASET,
    )

    plot_grid(
        gt=gt,
        pred_ws_plus=pred_ws_plus,
        target_cols=target_cols,
        time_index=time_index,
        output_dir=OUTPUT_DIR,
        dataset_name=DATASET,
    )

    plot_grouped(
        gt=gt,
        pred_ws_plus=pred_ws_plus,
        target_cols=target_cols,
        time_index=time_index,
        output_dir=OUTPUT_DIR,
        dataset_name=DATASET,
    )

    print(f"\n{'='*60}")
    print(f"[DONE] All plots saved to: {OUTPUT_DIR}")
    print(f"{'='*60}")


if __name__ == "__main__":
    main()

Loaded meta from: work/EUR/prepared_rabbitmq/meta.json
Time column: time
Target columns (10): ['cpu_limit', 'cpu_usage', 'lat50_ms', 'lat75_ms', 'lat95_ms', 'lat99_ms', 'min_ms', 'n', 'ram_limit_mb', 'ram_usage_mb']
Conditioning columns (6): ['t_norm', 'sin_day', 'cos_day', 'is_gap', 'time_since_last_obs', 'time_to_next_obs']

Data shapes: GT=(2612, 17), WaveStitch+=(2612, 17)
[INFO] Time sample: 1647991499.0 (type: float64)
[INFO] Detected Unix timestamp (seconds)
[INFO] Time range: 2022-03-22 23:24:59 to 2022-03-25 01:27:38
[INFO] Duration: 2 days 02:02:39

[PLOTTING] Individual feature plots...
[SAVED] work/EUR/generated_rabbitmq/plots/cpu_limit.png
[SAVED] work/EUR/generated_rabbitmq/plots/cpu_limit.pdf
[SAVED] work/EUR/generated_rabbitmq/plots/cpu_usage.png
[SAVED] work/EUR/generated_rabbitmq/plots/cpu_usage.pdf
[SAVED] work/EUR/generated_rabbitmq/plots/lat50_ms.png
[SAVED] work/EUR/generated_rabbitmq/plots/lat50_ms.pdf
[SAVED] work/EUR/generated_rabbitmq/plots/lat75_ms.png
[SAVED